[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/HyperCoast/blob/main/docs/examples/cesl.ipynb)

# Working with CESL Field Spectra

This notebook demonstrates how to use HyperCoast to search the Coastal Ecosystem Spectral Library ([CESL](https://speclib.nearview.net)), inspect site metadata, retrieve spectra, plot a selected sample, export CESL sites to GeoJSON, and visualize it on a map.

Credits to [Nearview](https://speclib.nearview.net/) for providing the CESL Spectral Library and its [REST API](https://docs.google.com/document/d/1ohgJ9wZnyHz4DeKPk99CeETg_fobKHV7eYADDRwzPYg/edit?tab=t.0).


In [ ]:
# %pip install "hypercoast[extra]"

In [ ]:
import json

import geopandas as gpd
import hypercoast

## Search the CESL catalog

Search the CESL catalog for a target species. The catalog query returns CESL sample IDs that can be used to fetch metadata and spectra.

In [ ]:
sample_ids = hypercoast.search_cesl(taxonomy="Fucus/vesiculosus")
len(sample_ids), sample_ids[:10]

## Inspect metadata for one CESL site

Each CESL sample includes site, sample, and acquisition metadata. HyperCoast flattens the CESL metadata schema so the values are easy to inspect.

In [ ]:
sample_id = sample_ids[0]
metadata = hypercoast.get_cesl_metadata(sample_id)

{
    key: metadata[key]
    for key in [
        "site_name",
        "sample_name",
        "sample_taxonomy_1",
        "location_lat",
        "location_lon",
        "measure_acquisition_date",
    ]
    if key in metadata
}

## Retrieve and plot a spectrum

Get the wavelength and reflectance values for the selected sample and plot the spectrum. You can use `x_range`, `y_range`, and `figsize` to suppress visible outliers.

In [ ]:
spectrum = hypercoast.get_cesl_spectrum(sample_id)
spectrum.head()

In [ ]:
hypercoast.plot_cesl_spectrum(
    sample_id,
    color="darkgreen",
    y_range=(0.0, 0.5),
    figsize=(12, 4),
)

![](https://github.com/user-attachments/assets/4d9e5875-6c61-4f7d-ad0a-645a17e98345)

## Export CESL sites to GeoJSON

Create a GeoJSON file containing all CESL sites. This requests the metadata for every sample in the CESL catalog, so it can take a little while to finish.

In [ ]:
geojson_path = "cesl_sites.geojson"
geojson = hypercoast.cesl_to_geojson(geojson_path, skip_errors=True)
len(geojson["features"]), geojson_path

In [ ]:
with open(geojson_path, encoding="utf-8") as file:
    preview = json.load(file)

preview["features"][0]

## Convert the GeoJSON to a GeoDataFrame

Use `GeoDataFrame.from_features(...)` when you already have an in-memory GeoJSON dictionary. Use `gpd.read_file(...)` only with a file path such as `geojson_path`.

In [ ]:
columns = [
    "sample_id",
    "site_name",
    "sample_subtype",
    "measure_acquisition_date",
    "location_lat",
    "location_lon",
    "geometry",
]
gdf = gpd.read_file(geojson_path)
gdf.crs = "EPSG:4326"
gdf = gdf[columns]
gdf.head()

## Visualize CESL sites on a map

Render CESL sites on an interactive map.

In [ ]:
gdf["sample_subtype"].unique()

In [ ]:
m = hypercoast.Map(center=(43.8, -69.85), zoom=8)
m.add_basemap("SATELLITE")
m.add_points_from_xy(
    gdf,
    x="location_lon",
    y="location_lat",
    color_column="sample_subtype",
    marker_colors=["green", "blue", "red"],
    add_legend=True,
)
m

![](https://github.com/user-attachments/assets/2a99b320-aa7e-496d-901d-c8857e6a5902)